<a href="https://colab.research.google.com/github/Rajapakshaminindu/Why-Your-Data-Runs-Slow-A-CPU-Cache-Study/blob/main/analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
"""
Cache Performance Analysis
===========================
Loads results.csv from the C experiments and produces:
  Chart 1 — Average time per experiment (bar chart)
  Chart 2 — All 8 runs per experiment (box + strip plot)
  Chart 3 — Speedup ratios (how much faster cache-friendly is)
  Chart 4 — Theory check: implied hit rate from Lecture 02 formula

Run from the analysis/ folder:
    python3 analysis.py

Output: saves 4 PNG files into analysis/charts/
"""

import os
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')                   # no display needed (headless)
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# ── 0. Setup ─────────────────────────────────────────────────────────────────

os.makedirs("charts", exist_ok=True)

COLORS = {
    "A_row_major":  "#1D9E75",   # teal   — cache-friendly
    "B_col_major":  "#D85A30",   # coral  — cache-unfriendly
    "C_sequential": "#1D9E75",   # teal   — cache-friendly
    "D_random":     "#D85A30",   # coral  — cache-unfriendly
}
LABELS = {
    "A_row_major":  "A: Row-major\n(cache-friendly)",
    "B_col_major":  "B: Col-major\n(cache-unfriendly)",
    "C_sequential": "C: Sequential\n(cache-friendly)",
    "D_random":     "D: Random\n(cache-unfriendly)",
}

plt.rcParams.update({
    "font.family":   "DejaVu Sans",
    "font.size":     11,
    "axes.spines.top":    False,
    "axes.spines.right":  False,
    "axes.grid":     True,
    "grid.alpha":    0.3,
    "grid.linestyle": "--",
    "figure.dpi":    150,
})

# ── 1. Load data ──────────────────────────────────────────────────────────────

df = pd.read_csv("results.csv")

ORDER = ["A_row_major", "B_col_major", "C_sequential", "D_random"]
df["experiment"] = pd.Categorical(df["experiment"], categories=ORDER, ordered=True)
df = df.sort_values("experiment")

summary = (df.groupby("experiment", observed=True)["time_ms"]
             .agg(mean="mean", std="std", median="median",
                  min="min", max="max")
             .round(3))

print("\n=== Summary Statistics (ms) ===")
print(summary.to_string())

# Speedup ratios
mean_A = summary.loc["A_row_major",  "mean"]
mean_B = summary.loc["B_col_major",  "mean"]
mean_C = summary.loc["C_sequential", "mean"]
mean_D = summary.loc["D_random",     "mean"]

speedup_BA = mean_B / mean_A
speedup_DC = mean_D / mean_C

print(f"\n  B is {speedup_BA:.1f}x slower than A (col-major vs row-major)")
print(f"  D is {speedup_DC:.1f}x slower than C (random vs sequential)")

# ── Chart 1: Average times bar chart ─────────────────────────────────────────

fig, ax = plt.subplots(figsize=(8, 5))

exps   = ORDER
means  = [summary.loc[e, "mean"]   for e in exps]
errors = [summary.loc[e, "std"]    for e in exps]
cols   = [COLORS[e]                for e in exps]
xlabs  = [LABELS[e]                for e in exps]

bars = ax.bar(xlabs, means, color=cols, width=0.55,
              yerr=errors, capsize=5, error_kw={"linewidth": 1.2})

# Annotate each bar with the mean value
for bar, mean in zip(bars, means):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + max(means) * 0.015,
            f"{mean:.1f} ms",
            ha="center", va="bottom", fontsize=10, fontweight="bold")

# Legend
patch_fast = mpatches.Patch(color="#1D9E75", label="Cache-friendly")
patch_slow = mpatches.Patch(color="#D85A30", label="Cache-unfriendly")
ax.legend(handles=[patch_fast, patch_slow], loc="upper left", frameon=False)

ax.set_ylabel("Average time (ms)", fontsize=11)
ax.set_title("Chart 1 — Average execution time by experiment\n"
             "Same work, different memory access patterns", fontsize=12)
ax.set_ylim(0, max(means) * 1.22)
fig.tight_layout()
fig.savefig("charts/chart1_avg_times.png")
plt.close(fig)
print("\nSaved: charts/chart1_avg_times.png")

# ── Chart 2: Box plot — all 8 runs ────────────────────────────────────────────

fig, ax = plt.subplots(figsize=(8, 5))

plot_data = [df[df["experiment"] == e]["time_ms"].values for e in ORDER]

bp = ax.boxplot(plot_data, patch_artist=True, widths=0.45,
                medianprops=dict(color="white", linewidth=2))

for patch, exp in zip(bp["boxes"], ORDER):
    patch.set_facecolor(COLORS[exp])
    patch.set_alpha(0.85)

# Overlay individual data points (strip plot style)
for i, (exp, data) in enumerate(zip(ORDER, plot_data), start=1):
    jitter = np.random.default_rng(seed=42).uniform(-0.12, 0.12, len(data))
    ax.scatter(np.full(len(data), i) + jitter, data,
               color=COLORS[exp], s=28, zorder=5, alpha=0.8, edgecolors="white",
               linewidths=0.5)

ax.set_xticks(range(1, 5))
ax.set_xticklabels([LABELS[e] for e in ORDER], fontsize=10)
ax.set_ylabel("Time (ms)", fontsize=11)
ax.set_title("Chart 2 — Distribution of all 8 runs per experiment\n"
             "Dots = individual runs, box = spread", fontsize=12)
fig.tight_layout()
fig.savefig("charts/chart2_boxplot.png")
plt.close(fig)
print("Saved: charts/chart2_boxplot.png")

# ── Chart 3: Speedup ratios ───────────────────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(9, 5))

for ax, (fast_exp, slow_exp, title) in zip(axes, [
    ("A_row_major",  "B_col_major",  "Matrix traversal\n(Exp B vs Exp A)"),
    ("C_sequential", "D_random",     "Array access\n(Exp D vs Exp C)"),
]):
    fast_mean = summary.loc[fast_exp, "mean"]
    slow_mean = summary.loc[slow_exp, "mean"]
    ratio = slow_mean / fast_mean

    bars = ax.bar(
        [LABELS[fast_exp], LABELS[slow_exp]],
        [fast_mean, slow_mean],
        color=[COLORS[fast_exp], COLORS[slow_exp]],
        width=0.5,
    )
    for bar, val in zip(bars, [fast_mean, slow_mean]):
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + slow_mean * 0.02,
                f"{val:.1f} ms",
                ha="center", va="bottom", fontsize=10, fontweight="bold")

    ax.set_title(f"{title}\nSpeedup = {ratio:.1f}x", fontsize=11)
    ax.set_ylabel("Average time (ms)", fontsize=10)
    ax.set_ylim(0, slow_mean * 1.25)

fig.suptitle("Chart 3 — Speedup: cache-friendly vs cache-unfriendly",
             fontsize=12, y=1.02)
fig.tight_layout()
fig.savefig("charts/chart3_speedup.png", bbox_inches="tight")
plt.close(fig)
print("Saved: charts/chart3_speedup.png")

# ── Chart 4: Theory check — Lecture 02 formula ───────────────────────────────
#
# T_avg = T1 + (1 - H) * T2
# From course Lecture 02:  T1 = 1 ns (L1 cache),  T2 = 60 ns (RAM)
#
# We measured time per experiment in ms over N accesses.
# Convert measured time to ns per access, then solve for H:
#   H = 1 - (T_avg_ns - T1) / T2
#
# Matrix exps:  N = 2048 * 2048 = 4,194,304 accesses
# Array  exps:  N = 16 * 1024 * 1024 = 16,777,216 accesses

T1, T2 = 1.0, 60.0   # nanoseconds

N_matrix = 2048 * 2048
N_array  = 16 * 1024 * 1024

access_counts = {
    "A_row_major":  N_matrix,
    "B_col_major":  N_matrix,
    "C_sequential": N_array,
    "D_random":     N_array,
}

hit_rates = {}
tavg_ns   = {}
for exp in ORDER:
    mean_ms = summary.loc[exp, "mean"]
    N       = access_counts[exp]
    t_ns    = (mean_ms * 1e6) / N        # convert ms → ns per access
    tavg_ns[exp] = t_ns
    H = 1.0 - (t_ns - T1) / T2
    H = max(0.0, min(1.0, H))            # clamp to [0, 1]
    hit_rates[exp] = H

print("\n=== Implied hit rates (Lecture 02 formula) ===")
for exp in ORDER:
    print(f"  {exp:18s}: T_avg = {tavg_ns[exp]:.2f} ns,  H = {hit_rates[exp]*100:.1f}%")

fig, ax = plt.subplots(figsize=(8, 5))

hr_vals = [hit_rates[e] * 100 for e in ORDER]
bars = ax.bar([LABELS[e] for e in ORDER], hr_vals,
              color=[COLORS[e] for e in ORDER], width=0.55)

for bar, hr in zip(bars, hr_vals):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.5,
            f"{hr:.1f}%",
            ha="center", va="bottom", fontsize=10, fontweight="bold")

ax.set_ylabel("Implied cache hit rate (%)", fontsize=11)
ax.set_ylim(0, 110)
ax.axhline(100, color="gray", linestyle="--", linewidth=0.8, alpha=0.5)
ax.set_title("Chart 4 — Implied hit rate H from Lecture 02 formula\n"
             r"$T_{avg} = T_1 + (1 - H) \times T_2$   "
             r"where $T_1 = 1\,\mathrm{ns}$, $T_2 = 60\,\mathrm{ns}$",
             fontsize=11)

patch_fast = mpatches.Patch(color="#1D9E75", label="Cache-friendly (high H)")
patch_slow = mpatches.Patch(color="#D85A30", label="Cache-unfriendly (lower H)")
ax.legend(handles=[patch_fast, patch_slow], loc="lower right", frameon=False)

fig.tight_layout()
fig.savefig("charts/chart4_hit_rate.png")
plt.close(fig)
print("Saved: charts/chart4_hit_rate.png")



print("\n=== Key findings ===")
print(f"  Matrix:  col-major is {speedup_BA:.1f}x SLOWER than row-major")
print(f"  Array:   random access is {speedup_DC:.1f}x SLOWER than sequential")
print(f"  Row-major implied hit rate  : {hit_rates['A_row_major']*100:.1f}%")
print(f"  Col-major implied hit rate  : {hit_rates['B_col_major']*100:.1f}%")
print(f"  Sequential implied hit rate : {hit_rates['C_sequential']*100:.1f}%")
print(f"  Random implied hit rate     : {hit_rates['D_random']*100:.1f}%")
print("\nAll charts saved to analysis/charts/")



=== Summary Statistics (ms) ===
                 mean    std   median      min      max
experiment                                             
A_row_major     2.081  0.159    2.082    1.905    2.337
B_col_major    35.307  1.511   34.698   34.275   38.803
C_sequential    9.940  0.233    9.880    9.640   10.357
D_random      199.993  4.069  198.384  196.660  208.864

  B is 17.0x slower than A (col-major vs row-major)
  D is 20.1x slower than C (random vs sequential)

Saved: charts/chart1_avg_times.png
Saved: charts/chart2_boxplot.png
Saved: charts/chart3_speedup.png

=== Implied hit rates (Lecture 02 formula) ===
  A_row_major       : T_avg = 0.50 ns,  H = 100.0%
  B_col_major       : T_avg = 8.42 ns,  H = 87.6%
  C_sequential      : T_avg = 0.59 ns,  H = 100.0%
  D_random          : T_avg = 11.92 ns,  H = 81.8%
Saved: charts/chart4_hit_rate.png

=== Key findings ===
  Matrix:  col-major is 17.0x SLOWER than row-major
  Array:   random access is 20.1x SLOWER than sequential
  Row-majo